# UHI Classification — Improved BaselineAn upgraded version of the Random-Forest baseline implementing the followingimprovements from the design memo:| # | Improvement                                           | Status in this notebook ||---|-------------------------------------------------------|-------------------------|| 1 | Combine Chile + Brazil training data                   | ✅ Built in, per-region toggles || 2 | Consistent GeoTIFF format across regions               | ✅ Enforced by configuration || 3 | Additional spectral indices (EVI, SAVI, MNDWI, …)     | ✅ Core indices; rest optional || 4 | Multi-scale neighbourhood features (30m / 100m / 250m) | ✅ One scale core; others optional || 5 | Building morphology (height, volume proxy, …)          | 🟡 Reads extra cols if available || 6 | Within-city standardisation                            | ✅ Core step || 7 | Stronger tabular models (XGBoost, LightGBM)            | ✅ Configurable `MODEL_CHOICE` || 8 | Open geospatial covariates (elevation, roads, …)       | ⛔ Optional; skipped unless data provided |**Design principle:** the notebook must run end-to-end with the data youcurrently have.  Every cell that depends on data you don't yet have is gatedby a configuration flag and silently skips when the flag is off.

In [ ]:
# ---------------------------------------------------------------------------# Clone the bc-II repo for the UHI CSV ground-truth files# ---------------------------------------------------------------------------!git clone https://github.com/chase-kusterer/bc-II.git%cd bc-II!cat .env_bin.txt > /dev/null!pip install -r /content/bc-II/requirements.txt

In [ ]:
# Mount Drive — source of GeoTIFFs and cached building-density filesfrom google.colab import drivedrive.mount('/content/drive')

In [ ]:
# Install extras for this notebook!pip install rioxarray xgboost lightgbm

In [ ]:
# Suppress non-critical warningsimport warningswarnings.filterwarnings('ignore')# File management and tabular dataimport osimport numpy as npimport pandas as pd# Geospatial raster / vector I/Oimport rioxarray as rxrimport rasterioimport geopandas as gpd# Visualisationimport matplotlib.pyplot as pltimport seaborn as sns# Modellingfrom sklearn.model_selection import train_test_split, StratifiedKFold, cross_validatefrom sklearn.ensemble import RandomForestClassifierfrom sklearn.metrics import (    accuracy_score, classification_report,    confusion_matrix, ConfusionMatrixDisplay,)# Optional stronger tabular models — loaded lazily in the training cell# depending on MODEL_CHOICE, so missing packages don't break the import cell.# Progress barfrom tqdm import tqdm

## ConfigurationEvery region-specific path and every "optional feature" switch lives in thiscell.  Running with just the data you currently have means leaving theoptional flags off — the pipeline automatically skips those feature blocks.### Memo idea #2 — Consistent GeoTIFF formatPoint the MBR/TBR path at the **same type** of GeoTIFF for every region.If Chile uses a median composite, Brazil and Freetown should too.  Mixing asingle-date scene with a median composite introduces confounding variancethat makes it impossible to separate model quality from imagery quality.

In [ ]:
# ===========================================================================# REGIONS TO INCLUDE IN TRAINING# ===========================================================================# Memo idea #1 — Combine Chile and Brazil.  Each region can be toggled off# individually (e.g., for ablation studies).INCLUDE_CHILE  = TrueINCLUDE_BRAZIL = True# ===========================================================================# INPUT FILE PATHS# ===========================================================================DRIVE_ROOT = "/content/drive/MyDrive/Business Challenge II"TIF_DIR    = f"{DRIVE_ROOT}/GeoTIFFs"BD_DIR     = f"{DRIVE_ROOT}/BuildingDensity"# --- Chile / Santiago ------------------------------------------------------CHILE_UHI_CSV       = "/content/bc-II/Data/sample_chile_uhi_data.csv"CHILE_GEOTIFF_PATH  = f"{TIF_DIR}/Santiago_MBR_Median_AllBands(2024-05-01TO2024-07-31).tiff"CHILE_DENSITY_CACHE = f"{BD_DIR}/Santiago_building_density_100m.parquet"# --- Brazil / Rio de Janeiro ----------------------------------------------BRAZIL_UHI_CSV       = "/content/bc-II/Data/sample_brazil_uhi_data.csv"  # adjust if named differentlyBRAZIL_GEOTIFF_PATH  = f"{TIF_DIR}/Rio_MBR_Median_AllBands(2023-05-01TO2023-07-31).tiff"BRAZIL_DENSITY_CACHE = f"{BD_DIR}/Rio_building_density_100m.parquet"# --- Validation region (Freetown) -----------------------------------------VAL_UHI_CSV       = "/content/bc-II/Data/validation_dataset.csv"VAL_GEOTIFF_PATH  = f"{TIF_DIR}/Freetown_MBR_Median_AllBands(2023-12-15TO2024-02-15).tiff"VAL_DENSITY_CACHE = f"{BD_DIR}/Freetown_building_density_100m.parquet"# ===========================================================================# OPTIONAL FEATURE FLAGS# Turn on only when you have the relevant data — the pipeline auto-skips# any block whose flag is False.# ===========================================================================# --- Memo idea #3 — Full spectral index set --------------------------------# True  : compute EVI, SAVI, MNDWI, BSI from the raw-bands GeoTIFF.# False : use only NDVI/NDBI/NDWI (sufficient if you only have the 3-band#         indices file).  The code auto-detects how many bands are present.USE_EXTENDED_INDICES = True# --- Memo idea #4 — Multi-scale neighbourhood features ---------------------# The BACKBONE uses the 100 m scale always.  Add 30 m and/or 250 m here for# extra signal.  Each additional scale multiplies feature count by ~index#.BUFFER_SCALES_M = [100]            # always include 100 m# BUFFER_SCALES_M = [30, 100, 250] # recommended once the pipeline is stable# --- Memo idea #5 — Extended building morphology ---------------------------# True  : read height/volume columns from the cached GeoParquet if present.# False : use `building_density_100m` only.USE_MORPHOLOGY_FEATURES = False    # flip to True once you rebuild the                                    # building density cache with extra cols# --- Memo idea #8 — Open geospatial covariates -----------------------------# Leave False until you have elevation/slope/roads rasters on Drive.USE_SPATIAL_COVARIATES = FalseSPATIAL_COVARIATES_DIR = f"{DRIVE_ROOT}/SpatialCovariates"   # future use# ===========================================================================# MODEL CHOICE# Memo idea #7 — Stronger tabular models# ===========================================================================# One of: "rf", "xgboost", "lightgbm"MODEL_CHOICE = "lightgbm"# ===========================================================================# OUTPUTS# ===========================================================================SUBMISSION_CSV = "/content/Predicted_Dataset.csv"# ---------------------------------------------------------------------------# Sanity-check required inputs only (optional inputs checked when used)# ---------------------------------------------------------------------------required = []if INCLUDE_CHILE:    required += [CHILE_UHI_CSV, CHILE_GEOTIFF_PATH, CHILE_DENSITY_CACHE]if INCLUDE_BRAZIL:    required += [BRAZIL_UHI_CSV, BRAZIL_GEOTIFF_PATH, BRAZIL_DENSITY_CACHE]required += [VAL_UHI_CSV, VAL_GEOTIFF_PATH, VAL_DENSITY_CACHE]missing = [p for p in required if not os.path.exists(p)]if missing:    print("Missing required inputs:")    for p in missing:        print(f"  {p}")    raise FileNotFoundError("Fix config paths before proceeding.")else:    print("All required inputs present.")    print(f"Model: {MODEL_CHOICE} | Extended indices: {USE_EXTENDED_INDICES} "          f"| Scales: {BUFFER_SCALES_M} | Morphology: {USE_MORPHOLOGY_FEATURES}")

## Feature Extraction HelpersThree helpers cover the feature-engineering backbone:* **`extract_buffered_features`** — memo idea #4.  For every point, loads a  square window of size `buffer_m` centered on the point from the GeoTIFF and  computes mean + std + min + max per band over that window.  This replaces  the old single-pixel `method="nearest"` extraction.* **`add_spectral_indices`** — memo idea #3.  Computes NDVI/NDBI/NDWI always,  plus EVI/SAVI/MNDWI/BSI when `USE_EXTENDED_INDICES` is on and the required  raw bands are available.* **`load_building_features`** — memo idea #5.  Reads the cached GeoParquet  and returns the density column always, plus morphology columns when they  exist and `USE_MORPHOLOGY_FEATURES` is on.

In [ ]:
# ===========================================================================# Helper — load any GeoTIFF and return an xarray DataArray with band names# ===========================================================================def open_geotiff(path):    """    Load a GeoTIFF.  If the file was written with band descriptions (the    All-Bands files from the upstream notebook set them), they are attached    to the xarray object for easy name-based access.    """    data = rxr.open_rasterio(path, masked=True)    # Attach band names from GeoTIFF tags when available    with rasterio.open(path) as src:        descs = src.descriptions        if descs and any(d for d in descs):            data = data.assign_coords(band_name=("band", list(descs)))    return data

In [ ]:
# ===========================================================================# Helper — per-point buffered pixel extraction# ===========================================================================def extract_buffered_features(geotiff_path, df, buffer_m, lat_col="Latitude", lon_col="Longitude"):    """    For every (lat, lon) in `df`, extract a square pixel window of side    `buffer_m` metres centred on the point, and return mean / std / min / max    of each band over that window.    Returns a DataFrame with columns named        {band}_{buffer_m}m_{stat}    where {stat} ∈ {mean, std, min, max}.    Assumes the GeoTIFF is in EPSG:4326.  At ≈111.32 km per degree of    latitude, `buffer_m` is converted to a degree half-width.    """    data = open_geotiff(geotiff_path)    band_names = (        list(data.coords["band_name"].values)        if "band_name" in data.coords        else [f"band{int(b)}" for b in data["band"].values]    )    # Half the window size in degrees (assumes CRS is EPSG:4326)    half_deg = (buffer_m / 2.0) / 111320.0    # Accumulate one dict per row    records = []    for lat, lon in tqdm(        zip(df[lat_col].values, df[lon_col].values),        total=len(df),        desc=f"Buffer {buffer_m}m features",    ):        # Slice a lat/lon box.  rioxarray y-axis is latitude (descending), so        # we use a slice spanning lat-half_deg .. lat+half_deg and rely on        # .sel to handle direction.        window = data.sel(            x=slice(lon - half_deg, lon + half_deg),            y=slice(lat + half_deg, lat - half_deg),  # y descending        )        # If the window is empty (point outside raster), fall back to nearest        if window.sizes["x"] == 0 or window.sizes["y"] == 0:            window = data.sel(x=lon, y=lat, method="nearest").expand_dims("x").expand_dims("y")        rec = {}        # Compute stats per band across the window's spatial dims        for i, bname in enumerate(band_names):            arr = window.isel(band=i).values.astype("float64")            valid = arr[~np.isnan(arr)] if arr.size else arr            if valid.size == 0:                m = s = mn = mx = np.nan            else:                m  = valid.mean()                s  = valid.std()                mn = valid.min()                mx = valid.max()            rec[f"{bname}_{buffer_m}m_mean"] = m            rec[f"{bname}_{buffer_m}m_std"]  = s            rec[f"{bname}_{buffer_m}m_min"]  = mn            rec[f"{bname}_{buffer_m}m_max"]  = mx        records.append(rec)    return pd.DataFrame(records)

In [ ]:
# ===========================================================================# Helper — compute spectral indices from buffered band statistics# ===========================================================================def add_spectral_indices(feat_df, buffer_m, extended=True):    """    Given a buffer-stats DataFrame produced by extract_buffered_features(),    add per-scale spectral indices built from band means.    Core indices (always attempted — require B03, B04, B08, B11):        NDVI = (B08 - B04) / (B08 + B04)        NDBI = (B11 - B08) / (B11 + B08)        NDWI = (B03 - B08) / (B03 + B08)    Extended indices (if `extended` and required bands present):        EVI   = 2.5 * (B08 - B04) / (B08 + 6*B04 - 7.5*B02 + 1)        SAVI  = 1.5 * (B08 - B04) / (B08 + B04 + 0.5)        MNDWI = (B03 - B11) / (B03 + B11)        BSI   = ((B11 + B04) - (B08 + B02)) / ((B11 + B04) + (B08 + B02))    When the source GeoTIFF is the 3-band indices file (bands named NDVI /    NDBI / NDWI rather than B01..B12), this function silently skips — the    indices are already present in the raw columns.    """    def c(name):        # Convenience accessor for the mean-stat column of a given band        col = f"{name}_{buffer_m}m_mean"        return feat_df[col] if col in feat_df.columns else None    b02, b03, b04, b08, b11 = c("B02"), c("B03"), c("B04"), c("B08"), c("B11")    # Core indices require B03, B04, B08, B11    if all(x is not None for x in (b03, b04, b08, b11)):        feat_df[f"NDVI_{buffer_m}m"] = (b08 - b04) / (b08 + b04 + 1e-9)        feat_df[f"NDBI_{buffer_m}m"] = (b11 - b08) / (b11 + b08 + 1e-9)        feat_df[f"NDWI_{buffer_m}m"] = (b03 - b08) / (b03 + b08 + 1e-9)    # Extended indices additionally need B02 for EVI and BSI    if extended and all(x is not None for x in (b02, b03, b04, b08, b11)):        feat_df[f"EVI_{buffer_m}m"]   = 2.5 * (b08 - b04) / (b08 + 6*b04 - 7.5*b02 + 1 + 1e-9)        feat_df[f"SAVI_{buffer_m}m"]  = 1.5 * (b08 - b04) / (b08 + b04 + 0.5 + 1e-9)        feat_df[f"MNDWI_{buffer_m}m"] = (b03 - b11) / (b03 + b11 + 1e-9)        feat_df[f"BSI_{buffer_m}m"]   = (            ((b11 + b04) - (b08 + b02)) / ((b11 + b04) + (b08 + b02) + 1e-9)        )    return feat_df

In [ ]:
# ===========================================================================# Helper — load building features from cached GeoParquet# ===========================================================================def load_building_features(cache_path, use_morphology=False):    """    Load a building-density GeoParquet produced by the shpFileReader notebook.    Returns a plain DataFrame containing only the building feature columns    (density always, morphology if requested and present) in the same row    order as the original CSV.    """    gdf = gpd.read_parquet(cache_path)    keep = ['building_density_100m']    if use_morphology:        # Memo idea #5 — include any morphology columns that were saved into        # the cache.  Silently ignore columns that aren't present so the        # pipeline still runs with a density-only cache.        morph_cols = [            'mean_height_100m', 'max_height_100m',            'footprint_ratio_100m', 'building_count_100m',            'built_volume_proxy_100m',        ]        keep += [c for c in morph_cols if c in gdf.columns]        print(f"  Morphology columns used: "              f"{[c for c in keep if c != 'building_density_100m']}")    return pd.DataFrame(gdf[keep])

## Per-Region Feature AssemblyFor each training region we run the same pipeline:1. Read the CSV of labelled points.2. Extract buffered band statistics at every configured scale.3. Compute spectral indices per scale.4. Load cached building features.5. Concatenate and tag rows with `region_id` for within-city standardisation.

In [ ]:
# ===========================================================================# Wraps the per-region assembly so the same code applies to Chile, Brazil,# and Freetown.# ===========================================================================def build_region_features(    uhi_csv,    geotiff_path,    density_cache,    region_id,    has_labels=True,):    """    Build the full feature DataFrame for one region.    Parameters    ----------    uhi_csv, geotiff_path, density_cache : str        Input paths for the region.    region_id : str        Label written into a `region` column — needed for within-city        standardisation (memo idea #6).    has_labels : bool        Training regions have UHI_Class; validation may not.    Returns    -------    pandas.DataFrame        CSV columns + per-scale buffer features + spectral indices +        building features + `region` column.    """    print(f"\n=== Building features for {region_id} ===")    df = pd.read_csv(uhi_csv).reset_index(drop=True)    print(f"  Points: {len(df)}")    # --- Memo idea #4 — per-scale buffered features ------------------------    all_scale_feats = []    for scale in BUFFER_SCALES_M:        feats = extract_buffered_features(geotiff_path, df, buffer_m=scale)        # --- Memo idea #3 — per-scale spectral indices ---------------------        feats = add_spectral_indices(feats, scale, extended=USE_EXTENDED_INDICES)        all_scale_feats.append(feats)    buffer_df = pd.concat(all_scale_feats, axis=1)    # --- Memo idea #5 — building morphology (density always) ---------------    bldg_df = load_building_features(density_cache, use_morphology=USE_MORPHOLOGY_FEATURES)    # Combine: original CSV cols + buffer features + building features    out = pd.concat(        [df.reset_index(drop=True),         buffer_df.reset_index(drop=True),         bldg_df.reset_index(drop=True)],        axis=1,    )    out["region"] = region_id    return out

In [ ]:
# ---------------------------------------------------------------------------# Assemble each training region, then concatenate on axis=0 (rows).# ---------------------------------------------------------------------------train_regions = []if INCLUDE_CHILE:    chile_df = build_region_features(        CHILE_UHI_CSV, CHILE_GEOTIFF_PATH, CHILE_DENSITY_CACHE,        region_id="chile", has_labels=True,    )    train_regions.append(chile_df)if INCLUDE_BRAZIL:    brazil_df = build_region_features(        BRAZIL_UHI_CSV, BRAZIL_GEOTIFF_PATH, BRAZIL_DENSITY_CACHE,        region_id="brazil", has_labels=True,    )    train_regions.append(brazil_df)if not train_regions:    raise ValueError("At least one training region must be enabled.")train_full = pd.concat(train_regions, axis=0, ignore_index=True)print(f"\nCombined training shape: {train_full.shape}")print("Class balance by region:")print(train_full.groupby("region")["UHI_Class"].value_counts())

## Within-City Standardisation (Memo Idea #6)Raw feature magnitudes transfer poorly across cities — a "high NDVI" inFreetown (tropical, rainy) is a different absolute value than a "high NDVI"in Santiago (semi-arid).  Converting each feature to a **z-score within itsregion** removes that systematic offset so the model learns *relative*patterns that transfer.This happens **before** the train/test split and **before** validationpreprocessing — critically, the validation region must be standardised using**its own** statistics, not the training statistics.  That's why `region` isa first-class column.

In [ ]:
# ===========================================================================# Within-city standardisation helper# ===========================================================================def standardise_within_region(df, feature_cols, region_col="region"):    """    Replace each value in `feature_cols` with its z-score computed within its    region group.  Returns a new DataFrame; non-feature columns are untouched.    """    out = df.copy()    for col in feature_cols:        # Standardise within group — subtract group mean, divide by group std        grp = out.groupby(region_col)[col]        out[col] = (out[col] - grp.transform("mean")) / (grp.transform("std") + 1e-9)    return out

In [ ]:
# ---------------------------------------------------------------------------# Identify the feature columns — anything that isn't metadata/label/geometry.# The assembly above put buffer stats, indices, and building features into# their own columns with systematic naming, so we can exclude the others.# ---------------------------------------------------------------------------META_COLS = {"Longitude", "Latitude", "UHI_Class", "region", "geometry"}feature_cols = [c for c in train_full.columns if c not in META_COLS]# Drop any rows containing NaN in predictors before standardisation so the# per-region mean/std statistics are clean.before = len(train_full)train_full = train_full.dropna(subset=feature_cols).reset_index(drop=True)print(f"Dropped {before - len(train_full)} rows with NaN features. "      f"Remaining: {len(train_full)}")# Apply within-region z-score standardisationtrain_std = standardise_within_region(train_full, feature_cols)print(f"Feature count after assembly + standardisation: {len(feature_cols)}")

## Train / Test SplitStratified on UHI_Class so both splits reflect the overall class mix.`random_state` is fixed for reproducibility.

In [ ]:
# ---------------------------------------------------------------------------# Prepare X, y and split.  `region` is kept in the full frame for diagnostics# but NOT passed to the model — models should not learn to predict based on# city identity.# ---------------------------------------------------------------------------X = train_std[feature_cols].valuesy = train_std["UHI_Class"].valuesX_train, X_test, y_train, y_test = train_test_split(    X, y, test_size=0.3, stratify=y, random_state=123,)print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")

## Model Training (Memo Idea #7)`MODEL_CHOICE` selects the algorithm.  All three handle class imbalance viaa `class_weight='balanced'` (RF) or equivalent (`scale_pos_weight` /sample weights for the gradient-boosting models).A 5-fold stratified CV estimate is printed before fitting the final modelso you have a generalisation number alongside the held-out test score.

In [ ]:
# ===========================================================================# Build the classifier according to MODEL_CHOICE# ===========================================================================def make_model(choice):    if choice == "rf":        return RandomForestClassifier(            n_estimators=500, max_depth=20, min_samples_leaf=2,            random_state=42, class_weight='balanced', n_jobs=-1,        )    if choice == "xgboost":        from xgboost import XGBClassifier        # XGBoost needs integer labels; we'll handle encoding at fit time.        return XGBClassifier(            n_estimators=500, max_depth=6, learning_rate=0.05,            subsample=0.9, colsample_bytree=0.9,            objective="multi:softmax",            eval_metric="mlogloss", verbosity=0,            random_state=42, n_jobs=-1,        )    if choice == "lightgbm":        from lightgbm import LGBMClassifier        return LGBMClassifier(            n_estimators=500, max_depth=-1, learning_rate=0.05,            num_leaves=63, subsample=0.9, colsample_bytree=0.9,            class_weight='balanced',            random_state=42, n_jobs=-1, verbose=-1,        )    raise ValueError(f"Unknown MODEL_CHOICE: {choice}")model = make_model(MODEL_CHOICE)print(f"Model: {type(model).__name__}")

In [ ]:
# ---------------------------------------------------------------------------# If the chosen model needs integer-encoded labels (XGBoost), encode here.# RandomForest and LightGBM both accept string labels directly.# ---------------------------------------------------------------------------from sklearn.preprocessing import LabelEncoderneeds_int_labels = MODEL_CHOICE == "xgboost"if needs_int_labels:    le = LabelEncoder()    y_train_fit = le.fit_transform(y_train)    y_test_fit  = le.transform(y_test)else:    le = None    y_train_fit = y_train    y_test_fit  = y_test

In [ ]:
# ---------------------------------------------------------------------------# 5-fold stratified CV — generalisation estimate on the training split only# ---------------------------------------------------------------------------cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)cv_results = cross_validate(    make_model(MODEL_CHOICE),    X_train, y_train_fit,    cv=cv,    scoring=['accuracy', 'f1_macro'],    n_jobs=-1,)print(f"CV accuracy : {cv_results['test_accuracy'].mean():.4f} "      f"± {cv_results['test_accuracy'].std():.4f}")print(f"CV macro-F1 : {cv_results['test_f1_macro'].mean():.4f} "      f"± {cv_results['test_f1_macro'].std():.4f}")

In [ ]:
# ---------------------------------------------------------------------------# Fit the final model on the full training split# ---------------------------------------------------------------------------model.fit(X_train, y_train_fit)print("Training complete.")

## Out-of-Sample Evaluation

In [ ]:
# ---------------------------------------------------------------------------# Predict on the held-out test split# ---------------------------------------------------------------------------y_pred_fit = model.predict(X_test)y_pred = le.inverse_transform(y_pred_fit) if le is not None else y_pred_fitacc = accuracy_score(y_test, y_pred)print(f"Test accuracy: {acc:.4f}\n")print(classification_report(y_test, y_pred))

In [ ]:
# ---------------------------------------------------------------------------# Confusion matrix# ---------------------------------------------------------------------------labels = sorted(np.unique(y_test))cm = confusion_matrix(y_test, y_pred, labels=labels)ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels).plot(cmap='Blues')plt.title(f'Confusion Matrix — {MODEL_CHOICE}')plt.grid(False)plt.show()

In [ ]:
# ---------------------------------------------------------------------------# Feature importance — top 20 features by the model's native importance# ---------------------------------------------------------------------------if hasattr(model, "feature_importances_"):    importances = model.feature_importances_    order = np.argsort(importances)[::-1][:20]  # top 20    plt.figure(figsize=(9, 6))    plt.title(f"Top 20 Feature Importances — {MODEL_CHOICE}")    plt.barh(range(len(order)), importances[order][::-1], align='center')    plt.yticks(range(len(order)),               [feature_cols[i] for i in order[::-1]])    plt.tight_layout()    plt.show()else:    print(f"{type(model).__name__} does not expose feature_importances_.")

## Apply to Validation Region (Freetown)Same pipeline — extract features, standardise using Freetown's OWNwithin-region statistics (critical: do NOT use training stats), predict.

In [ ]:
# ---------------------------------------------------------------------------# Build validation features with identical structure# ---------------------------------------------------------------------------val_df = build_region_features(    VAL_UHI_CSV, VAL_GEOTIFF_PATH, VAL_DENSITY_CACHE,    region_id="freetown", has_labels=False,)# Validate that every training feature column is present in the validation# frame — otherwise the model's predict() would fail.missing_cols = [c for c in feature_cols if c not in val_df.columns]if missing_cols:    raise ValueError(f"Validation frame is missing columns: {missing_cols}")# Drop NaN rows (to match the cleaning applied to training data)val_df = val_df.dropna(subset=feature_cols).reset_index(drop=True)# Within-city standardisation for the validation region, using ITS OWN statsval_std = standardise_within_region(val_df, feature_cols)X_val = val_std[feature_cols].valuesprint(f"X_val: {X_val.shape}")

In [ ]:
# ---------------------------------------------------------------------------# Predict and build the submission frame# ---------------------------------------------------------------------------val_pred_fit = model.predict(X_val)val_pred = le.inverse_transform(val_pred_fit) if le is not None else val_pred_fitsubmission_df = pd.DataFrame({    'Longitude': val_std['Longitude'].values,    'Latitude' : val_std['Latitude'].values,    'UHI_Class': val_pred,})print("Predicted class distribution:")print(submission_df['UHI_Class'].value_counts())submission_df.head()

In [ ]:
# Save submission CSVsubmission_df.to_csv(SUBMISSION_CSV, index=False)print(f"Submission saved to: {SUBMISSION_CSV}")

## Notes on Deferred ImprovementsThe following memo ideas are **not implemented** in this notebook becausethey require data or code you don't yet have.  They're called out here so youcan plan the upstream work.### Memo idea #5 — Extended building morphology (partially deferred)Requires re-running the `shpFileReader` notebook with additional columnswritten into the cached GeoParquet.  Suggested columns to compute alongside`building_density_100m`:```python# Add to compute_building_density() in shpFileReader:#   mean_height_100m        = buildings_in_buffer['height'].mean()#   max_height_100m         = buildings_in_buffer['height'].max()#   building_count_100m     = len(buildings_in_buffer)#   footprint_ratio_100m    = buildings_in_buffer.geometry.area.sum() / area_m2#   built_volume_proxy_100m = (buildings_in_buffer.geometry.area *#                              buildings_in_buffer['height']).sum() / area_m2```Once the cache has those columns, set `USE_MORPHOLOGY_FEATURES = True`.### Memo idea #8 — Open geospatial covariates (fully deferred)Elevation (SRTM / Copernicus DEM), slope (derived from elevation), distanceto coastline, road density (from OpenStreetMap) are all publicly availablebut require separate preprocessing pipelines.  When you have them as rastersaligned to the AOI, extend `build_region_features` with a call like:```pythoncovariates = extract_buffered_features(    SPATIAL_COVARIATES_DIR + "/elevation.tiff", df, buffer_m=100)```and concatenate into `out`.  Set `USE_SPATIAL_COVARIATES = True` once thedata is in place.### Multi-scale expansion (partial)`BUFFER_SCALES_M = [100]` is the current backbone.  Once the pipeline isvalidated, move to `[30, 100, 250]` for the full multi-scale set.  Expectfeature count to roughly triple and runtime to scale linearly.